##### q0 Base --> X
##### q1 Hombro --> +180
##### q2 Codo --> X
##### q3 Muñeca 1 --> +35
##### q4 Muñeca 2 --> +90
##### q5 Muñeca 3 --> +0

In [48]:
import rtde_control
import rtde_receive
import rtde_io
import numpy as np

ROBOT_IP = "192.168.19.128"

pos_inicial_grados = [0.0, 180.0, -125.0, 35.0, 90.0, 0.0]
pos_inicial = np.radians(pos_inicial_grados).tolist()

rtde_c = None
rtde_r = None
rtde_io_ = None

try:
    rtde_c = rtde_control.RTDEControlInterface(ROBOT_IP)
    rtde_r = rtde_receive.RTDEReceiveInterface(ROBOT_IP)
    rtde_io_ = rtde_io.RTDEIOInterface(ROBOT_IP)

    rtde_c.moveJ(pos_inicial, speed=0.5, acceleration=0.5)

finally:
    if rtde_io_ is not None:
        rtde_io_.disconnect()
    if rtde_c is not None:
        rtde_c.disconnect()
    if rtde_r is not None:
        rtde_r.disconnect()

In [10]:
import rtde_control
import rtde_receive
import rtde_io
import numpy as np
from scipy.optimize import minimize

ROBOT_IP = "192.168.19.128"

# Posición inicial (grados)
pos_inicial_grados = [0.0, 180.0, -125.0, 35.0, 90.0, 0.0]
pos_inicial = np.radians(pos_inicial_grados).tolist()

# Parámetros DH del UR5
d1, a2, a3, d4, d5, d6 = 0.1625, -0.425, -0.3922, 0.1333, 0.0997, 0.0996

def dh_matrix(theta, d, a, alpha):
    return np.array([
        [np.cos(theta), -np.sin(theta)*np.cos(alpha),  np.sin(theta)*np.sin(alpha), a*np.cos(theta)],
        [np.sin(theta),  np.cos(theta)*np.cos(alpha), -np.cos(theta)*np.sin(alpha), a*np.sin(theta)],
        [0,              np.sin(alpha),                np.cos(alpha),               d],
        [0,              0,                            0,                           1]
    ])

def forward_kinematics(q):
    T = np.eye(4)
    params = [
        (q[0], d1, 0, np.pi/2),
        (q[1], 0, a2, 0),
        (q[2], 0, a3, 0),
        (q[3], d4, 0, np.pi/2),
        (q[4], d5, 0, -np.pi/2),
        (q[5], d6, 0, 0)
    ]
    for theta, d, a, alpha in params:
        T = T @ dh_matrix(theta, d, a, alpha)
    return T

def ik_free_q0_q2_q3(target_xyz, q1_fixed, q4_fixed, q5_fixed):
    x_t, y_t, z_t = target_xyz
    def objective(vars):
        q0, q2, q3 = vars
        q = [q0, q1_fixed, q2, q3, q4_fixed, q5_fixed]
        T = forward_kinematics(q)
        x, y, z = T[0,3], T[1,3], T[2,3]
        return (x - x_t)**2 + (y - y_t)**2 + (z - z_t)**2
    res = minimize(objective, x0=[0.0, -2.0, 0.5], bounds=[
        (-np.pi, np.pi),    # q0
        (-np.pi, np.pi),    # q2
        (-np.pi, np.pi)     # q3
    ], tol=1e-6)
    if res.success and res.fun < 1e-4:
        return [res.x[0], q1_fixed, res.x[1], res.x[2], q4_fixed, q5_fixed]
    return None

rtde_c = None
rtde_r = None
rtde_io_ = None

try:
    rtde_c = rtde_control.RTDEControlInterface(ROBOT_IP)
    rtde_r = rtde_receive.RTDEReceiveInterface(ROBOT_IP)
    rtde_io_ = rtde_io.RTDEIOInterface(ROBOT_IP)

    # Ir a posición inicial
    rtde_c.moveJ(pos_inicial, speed=0.5, acceleration=0.5)

    # Obtener pose inicial y Z
    pose_inicial = rtde_r.getActualTCPPose()
    x0, y0, z0 = pose_inicial[0], pose_inicial[1], pose_inicial[2]
    rx, ry, rz = pose_inicial[3], pose_inicial[4], pose_inicial[5]

    # Matriz de rotación (ZYX)
    crx, srx = np.cos(rx), np.sin(rx)
    cry, sry = np.cos(ry), np.sin(ry)
    crz, srz = np.cos(rz), np.sin(rz)
    R = np.array([
        [cry*crz, -crx*srz + srx*sry*crz,  srx*srz + crx*sry*crz],
        [cry*srz,  crx*crz + srx*sry*srz, -srx*crz + crx*sry*srz],
        [-sry,     srx*cry,                crx*cry]
    ])

    # Dirección hacia adelante = -Y del TCP
    dir_forward = -R[:, 1]

    # Nueva posición: 20 cm hacia adelante, misma Z
    target_pos = np.array([x0, y0, z0]) + 0.20 * dir_forward
    target_pos[2] = z0  # mantener Z

    # Resolver IK con solo q0, q2, q3 libres
    q1_f = np.radians(180.0)
    q4_f = np.radians(90.0)
    q5_f = 0.0

    q_sol = ik_free_q0_q2_q3(target_pos, q1_f, q4_f, q5_f)

    if q_sol is not None:
        rtde_c.moveJ(q_sol, speed=0.3, acceleration=0.3)
        Qfinal = rtde_r.getActualQ()
        print("Posición final (grados):", np.degrees(Qfinal))
    else:
        print("❌ No se encontró solución.")

finally:
    if rtde_io_ is not None: rtde_io_.disconnect()
    if rtde_c is not None: rtde_c.disconnect()
    if rtde_r is not None: rtde_r.disconnect()

Posición final (grados): [  16.57621809  180.         -112.22606611   42.97731198   90.
    0.        ]


In [ ]:
import rtde_control
import rtde_receive
import rtde_io
import numpy as np
from scipy.optimize import minimize

ROBOT_IP = "192.168.19.128"

pos_inicial_grados = [0.0, 180.0, -125.0, 35.0, 90.0, 0.0]
pos_inicial = np.radians(pos_inicial_grados).tolist()

# Parámetros Denavit Hartenberg UR5
d1, a2, a3, d4, d5, d6 = 0.1625, -0.425, -0.3922, 0.1333, 0.0997, 0.0996

def forward_kinematics(q):
    from math import cos, sin, pi
    def dh(theta, d, a, alpha):
        return np.array([
            [cos(theta), -sin(theta)*cos(alpha),  sin(theta)*sin(alpha), a*cos(theta)],
            [sin(theta),  cos(theta)*cos(alpha), -cos(theta)*sin(alpha), a*sin(theta)],
            [0,           sin(alpha),             cos(alpha),            d],
            [0,           0,                      0,                     1]
        ])
    T = np.eye(4)
    for (th, d, a, al) in [(q[0], d1, 0, pi/2), (q[1], 0, a2, 0), (q[2], 0, a3, 0),
                           (q[3], d4, 0, pi/2), (q[4], d5, 0, -pi/2), (q[5], d6, 0, 0)]:
        T = T @ dh(th, d, a, al)
    return T

rtde_c = None
rtde_r = None
rtde_io_ = None

try:
    rtde_c = rtde_control.RTDEControlInterface(ROBOT_IP)
    rtde_r = rtde_receive.RTDEReceiveInterface(ROBOT_IP)
    rtde_io_ = rtde_io.RTDEIOInterface(ROBOT_IP)

    rtde_c.moveJ(pos_inicial, speed=0.5, acceleration=0.5)

    # Valores fijos
    q0_f = 0.0
    q1_f = np.radians(180.0)
    q4_f = np.radians(90.0)
    q5_f = 0.0

    # Pose inicial
    T0 = forward_kinematics(pos_inicial)
    x0, y0, z0 = T0[0,3], T0[1,3], T0[2,3]
    print(f"Posición inicial: x={x0:.3f}, y={y0:.3f}, z={z0:.3f}")

    # Nueva Y: 30 cm hacia adelante
    y_target = y0 + 0.3
    z_margin = 0.005  # ±5 mm

    def objective(vars):
        q2, q3 = vars
        q = [q0_f, q1_f, q2, q3, q4_f, q5_f]
        T = forward_kinematics(q)
        x, y, z = T[0,3], T[1,3], T[2,3]
        pos_error = (x - x0)**2 + (y - y_target)**2
        z_error = max(0, abs(z - z0) - z_margin)**2  # solo penaliza si fuera del margen
        return pos_error + 1000 * z_error

    q2_init = pos_inicial[2]
    q3_init = pos_inicial[3]

    res = minimize(objective, x0=[q2_init, q3_init],
                   bounds=[(-2.6, -0.4), (0.0, 1.2)],
                   method='L-BFGS-B', tol=1e-9)

    print(f"Optimización: éxito={res.success}, error_total={res.fun:.2e}")

    if res.success:
        # Verificar Z manualmente
        q_test = [q0_f, q1_f, res.x[0], res.x[1], q4_f, q5_f]
        T_test = forward_kinematics(q_test)
        z_test = T_test[2,3]
        if abs(z_test - z0) <= z_margin + 0.001:  # tolerancia pequeña
            q_sol = [q0_f, q1_f, float(res.x[0]), float(res.x[1]), q4_f, q5_f]
            rtde_c.moveJ(q_sol, speed=0.2, acceleration=0.3)
            print(f"✅ Movido a y={T_test[1,3]:.3f}, z={z_test:.3f} (dentro de margen)")
        else:
            print(f"❌ Z fuera de margen: {z_test:.4f} vs {z0:.4f}")
    else:
        print("❌ Optimización fallida.")

finally:
    if rtde_io_ is not None: rtde_io_.disconnect()
    if rtde_c is not None: rtde_c.disconnect()
    if rtde_r is not None: rtde_r.disconnect()

Posición inicial: x=0.300, y=-0.133, z=-0.258
Optimización: éxito=True, error_total=4.90e-01
✅ Movido a y=-0.133, z=-0.258 (dentro de margen)


In [44]:
import rtde_control
import rtde_receive
import numpy as np

ROBOT_IP = "192.168.19.128"

# Tu posición inicial (en grados)
pos_inicial_grados = [0.0, 180.0, -125.0, 35.0, 90.0, 0.0]
pos_inicial = np.radians(pos_inicial_grados).tolist()

rtde_c = rtde_control.RTDEControlInterface(ROBOT_IP)
rtde_r = rtde_receive.RTDEReceiveInterface(ROBOT_IP)

try:
    # 1. Ir a la posición inicial articular
    rtde_c.moveJ(pos_inicial, speed=0.5, acceleration=0.5)

    # 2. Leer la pose actual del TCP (justo después de llegar)
    actual_tcp_pose = rtde_r.getActualTCPPose()  # [x, y, z, rx, ry, rz]
    x0, y0, z0 = actual_tcp_pose[0], actual_tcp_pose[1], actual_tcp_pose[2]

    # 3. Definir orientación "mirando hacia abajo"
    # En convención UR: Rx = π, Ry = 0, Rz = 0
    orientacion_abajo = [np.pi, 0.0, 0.0]

    # 4. Nueva pose: misma Z, orientación fija hacia abajo
   # pose_abajo = [x0, y0, z0] + orientacion_abajo

    # 5. Mover en línea recta a esa pose (esto ajustará las articulaciones automáticamente)
    #rtde_c.moveL(pose_abajo, speed=0.2, acceleration=0.2)

    # ✅ A partir de ahora, cualquier movimiento debe usar [x, y, z0, π, 0, 0]
   # pose_mover_xy = [x0 + 0.3, y0, z0, np.pi, 0.0, 0.0]
    #rtde_c.moveL(pose_mover_xy, speed=0.2, acceleration=0.2)

finally:
    rtde_c.disconnect()
    rtde_r.disconnect()

In [ ]:
import rtde_control
import rtde_receive
import numpy as np

ROBOT_IP = "192.168.19.128"

# Posición inicial (tu pose)
pos_inicial_grados = [0.0, 180.0, -125.0, 35.0, 90.0, 0.0]
pos_inicial = np.radians(pos_inicial_grados).tolist()

rtde_c = rtde_control.RTDEControlInterface(ROBOT_IP)
rtde_r = rtde_receive.RTDEReceiveInterface(ROBOT_IP)

try:
    # Ir a la posición inicial
    rtde_c.moveJ(pos_inicial, speed=0.5, acceleration=0.5)
    pos_inicialxyz = rtde_r.getActualTCPPose()
    print(f"Posición inicial TCP: x={pos_inicialxyz[0]:.3f}, y={pos_inicialxyz[1]:.3f}, z={pos_inicialxyz[2]:.3f}")


    # Leer pose actual
    pose = rtde_r.getActualTCPPose()
    x0, y0, z0 = pose[0], pose[1], pose[2]
    print(f"Altura inicial Z: {z0:.4f} m")

    # ✅ NUEVA POSE: solo cambiamos q2 y q3, q1 sigue = 180°
    # Ajusta q2 y q3 para mover el TCP en X, manteniendo Z aproximadamente constante
    # Esto requiere prueba y error, o usar valores precalculados

    # Ejemplo: mover 20 cm en +X (adelante)
    # Estos valores son aproximados; debes ajustarlos según tu robot
    nueva_pos_grados = [0.0, 180.0, -105.00, 35.0, 90.0, 0.0]  # q2 más negativo = brazo más extendido
    nueva_pos_rad = np.radians(nueva_pos_grados).tolist()

    rtde_c.moveJ(nueva_pos_rad, speed=0.3, acceleration=0.3)

    # Verificar nueva pose
    pose2 = rtde_r.getActualTCPPose()
    print(f"Nueva pose -> X: {pose2[0]:.3f}, Y: {pose2[1]:.3f}, Z: {pose2[2]:.3f}")

finally:
    rtde_c.disconnect()
    rtde_r.disconnect()

Posición inicial TCP: x=0.300, y=-0.133, z=-0.258
Altura inicial Z: -0.2584 m
Nueva pose -> X: 0.451, Y: -0.133, Z: -0.276


# ESTE SI ES

In [4]:
import rtde_control
import rtde_receive
import numpy as np
from scipy.spatial.transform import Rotation as R

#ROBOT_IP = "192.168.19.128" #simulador
ROBOT_IP = "169.254.129.110" #robot real

# Define aquí tus 4 esquinas (x, y, z en metros)
esquinas_cartesianas = [
    [0.161, -0.781, -0.277],   # esquina A
    [-0.383, -0.700, -0.277],  # esquina B
    [-0.394, -0.269, -0.277],  # esquina C
    [0.147, -0.361, -0.277],   # esquina D
]

# ángulos de rotación [π, 0, 0]
rotacion_abajo = R.from_euler('xyz', [np.pi, 0, 0]).as_rotvec().tolist()

# Posición inicial
pos_inicial_grados = [-87, -151, -106, -11, 90, 183]
pos_inicial = np.radians(pos_inicial_grados).tolist()

rtde_c = rtde_control.RTDEControlInterface(ROBOT_IP)
rtde_r = rtde_receive.RTDEReceiveInterface(ROBOT_IP)

try:
    # Ir a la posición inicial
    rtde_c.moveJ(pos_inicial, speed=0.5, acceleration=0.5)
    pos_inicialxyz = rtde_r.getActualTCPPose()
    print(f"Posición inicial TCP: x={pos_inicialxyz[0]:.3f}, y={pos_inicialxyz[1]:.3f}, z={pos_inicialxyz[2]:.3f}")

    for i, (x, y, z) in enumerate(esquinas_cartesianas):
        print(f"\nMoviendo a esquina {i+1}: x={x}, y={y}, z={z}")
        
        # Crear pose: [x, y, z, rx, ry, rz]
        pose = [x, y, z] + rotacion_abajo
        
        # Mover en línea recta (TCP mira siempre hacia abajo)
        rtde_c.moveL(pose, speed=0.2, acceleration=0.2)

    #movimiento terminado, volver a la posición inicial
    rtde_c.moveJ(pos_inicial, speed=0.5, acceleration=0.5)


finally:
    rtde_c.disconnect()
    rtde_r.disconnect()

Posición inicial TCP: x=-0.103, y=-0.572, z=-0.140

Moviendo a esquina 1: x=0.161, y=-0.781, z=-0.277

Moviendo a esquina 2: x=-0.383, y=-0.7, z=-0.277

Moviendo a esquina 3: x=-0.394, y=-0.269, z=-0.277

Moviendo a esquina 4: x=0.147, y=-0.361, z=-0.277
